In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.utils import murmurhash3_32

#### 自先前的電影評論 csv 壓縮檔以資料串流(Data Streams)逐列讀出

In [2]:
import lzma
import csv
def stream_csv(path):
    with lzma.open(path, 'rt', encoding='utf-8') as f:
        reader = csv.reader(f)
        next(reader)  # 忽略第0列
        for line in reader:
            text, label = line[2], int(line[1])
            yield text, label

## 對資料串流小批量(至少1筆)取得訓練樣本

In [3]:
def getMinibatch(row,size):
    X,y = [],[]
    try:
        for _ in range(size):
            text, label = next(row)
            X.append(text)
            y.append(label)
    except StopIteration:
        print("文件讀取結束")
    finally:
        return X,y

#### 數值化向量及產生csr 稀疏矩陣
        HashingVectorizer的transform將文字向量轉換為數值向量
        SGDClassifier隨機梯度下降法分類器，以羅吉斯損失函數進行增量學習(partial_fit)

In [128]:
stream_line=stream_csv("./data/movie_data_zh_jieba.csv.xz")
num_rows = 48811    # 訓練樣本總筆數
batch_size = 2000   # 增量學習隻小批量大小
test_size=0.01      # 測試集占比
vect = HashingVectorizer(decode_error='ignore', 
                         n_features=2**21,
                         binary=False,
                         alternate_sign=False,
                         norm="l2",
                         preprocessor=None)
sgdclf = SGDClassifier(loss='log_loss', random_state=1)     # 損失函數使用羅吉斯迴歸
classes = np.array([0, 1])
for _ in range(int(num_rows/batch_size*(1-test_size))):
    X_train, y_train = getMinibatch(row=stream_line,size=batch_size) 
    if not X_train:
        break
    X_train = vect.transform(X_train)    
    sgdclf.partial_fit(X_train, y_train, classes=classes)

### 模型經訓練後進行準確率驗證
**將測試資料集數字化之後，使用模型參數進行準確率預估：**

        使用分類器的predict
        或直接使用分類器的score比較測試集準確率

In [130]:
print(f'模型參數:{sgdclf.coef_}')
X_test, y_test = getMinibatch(row=stream_line,size=5000)
X_test = vect.transform(X_test)
y_pred_test=sgdclf.predict(X_test)
print("測試集準確率:",accuracy_score(y_test, y_pred_test))
print('測試集準確率: %.3f' % sgdclf.score(X_test, y_test))

模型參數:[[ 0.          0.          0.         ...  0.          0.00162065
  -0.01059429]]


文件讀取結束
測試集準確率: 0.8508014796547472
測試集準確率: 0.851


# CSR矩陣資料結構(小批次X_train)

In [131]:
print(f'數值陣列(values): {X_train.data}')
print(f'行索引陣列(col_indices): {X_train.indices}')
print(f'列指標陣列(row_ptr): {X_train.indptr}')

數值陣列(values): [0.07053456 0.07053456 0.07053456 ... 0.10259784 0.10259784 0.41039134]
行索引陣列(col_indices): [  50396   59930   98505 ... 2008494 2045361 2087943]
列指標陣列(row_ptr): [     0     94    146 ... 158626 158769 158828]


# 示範自樣本資料集的第一筆第一個詞「萬聖節」：
    設定HashingVectorizer超參數(不進行歸一化(norm=None) 該詞的數值)，將其轉換為數字向量的csr矩陣    
    在crs稀疏矩陣的「列指標」與「行索引」下取回其值       

In [132]:
import copy
stream_line = stream_csv("./data/movie_data_zh_jieba.csv.xz")
X_part, y_part = getMinibatch(row=stream_line,size=1)   # 只取一筆樣本(第一筆)
print(X_part)
word = X_part[0].split()[0]     #  第一個詞
print(word)
vect1 = copy.copy(vect)
vect1.set_params(norm=None)     # 不進行歸一化
n_features = vect1.get_params()['n_features']
csr_obj = vect1.transform([word])   # 將單一個詞轉換為數字向量的稀疏矩陣
dense = csr_obj.todense()
print(f'稠密矩陣{dense.shape}: {dense}')
print(f'{word} hashing index: {abs(murmurhash3_32(word))%n_features}')
print(f'values(數值陣列): {csr_obj.data}')
print(f'col_indices(行索引陣列): {csr_obj.indices}')
print(f'row_ptr(列指標陣列): {csr_obj.indptr}')
print(f'{word} value: {csr_obj[0,1230375]}')

['萬聖節 惡作劇 之夜 青少年 瑪莎 克斯 利瑪吉 格蕾絲 康涅狄格州 格林威治 貝勒港 高級區 二十二年 殺案 偵破 作家 福爾曼 克里斯托弗 梅洛 一位 前洛杉磯 偵探 因在 辛普森 案審判 中作 偽證 名譽 掃地 搬到 愛達 荷州 決定 調查 這起 案件 史蒂芬 克斯 安德魯 切爾 辦案 目的 一本 當地 局促不安 不歡 他們 年代 負責 調查 退休 偵探 史蒂夫 卡羅爾 伯特 福斯特 支持 他們 發現 罪犯 一個 權力 金錢 網絡來 掩蓋 格林威治 殺案 這是 一部 電視 電影 講述 一名 富有 青少年 謀殺 一名 女孩 真實 故事 女孩 母親 甘迺迪 家族 強大而 富有 家族 利用 他們 影響力 掩蓋 這起謀 殺案 二十多年 一位 史努比 偵探 定罪 偽證者 恥辱 揭露 這起 可怕 罪行 犯下 劇本 同時 展示 對馬克 調查 瑪莎 最後幾天 但戲 劇化 缺乏 情感 投票 七個 標題 巴西']
萬聖節
稠密矩陣(1, 2097152): [[0. 0. 0. ... 0. 0. 0.]]
萬聖節 hashing index: 1230375
values(數值陣列): [1.]
col_indices(行索引陣列): [1230375]
row_ptr(列指標陣列): [0 1]
萬聖節 value: 1.0
